# Reset Shewhart Benchmark

This notebook demonstrates the reset online benchmark API with a Shewhart control chart.

Unlike the no-reset benchmark, `OnlineResetBenchmark` works with already configured `OnlineResetDetector` instances and evaluates arbitrary multiple-run metrics on whole timeseries runs. The result of `get_metrics_for(entries, metric)` is a dictionary: `detector_description -> metric_value`.

Sections:
1. Configuration and dataset preparation
2. Interactive dataset preview
3. Reset benchmark metric tables
4. Metric plots by threshold
5. Single-run trace inspection

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipywidgets import Dropdown, VBox, interactive_output

from pysatl_cpd.algorithms.online import ShewhartControlChart
from pysatl_cpd.analysis.metrics.multiple_run.classification import ClassificationReport
from pysatl_cpd.analysis.metrics.multiple_run.online import ARLMetric, MeanDelayMetric, MedianDelayMetric
from pysatl_cpd.analysis.visualization import DrawBackend, OnlineCpdPlotter
from pysatl_cpd.benchmark.online.reset import OnlineResetBenchmark, OnlineResetBenchmarkEntry
from pysatl_cpd.benchmark.registry import BenchmarkRegistry
from pysatl_cpd.core.online import OnlineResetDetector
from pysatl_cpd.data.generator import preset_dataset
from pysatl_cpd.data.providers.transformers import ColumnsSelectorTransformer

## Configuration

In [2]:
BENCHMARK_PRESET = "mean_shifts"
FEATURE = "feature_0"
N_SERIES = 20
SEED = 42

LEARNING_PERIOD_SIZE = 100
WINDOW_SIZE = 50

THRESHOLDS = np.linspace(2.0, 6.0, 9)
SKIP_PERIOD = 20
MAX_RUNLENGTH = None
ERROR_MARGIN = (0, 100)
MAX_DELAY = ERROR_MARGIN[1]

feature_transformer = ColumnsSelectorTransformer(columns=[FEATURE])

In [3]:
def make_detector(threshold: float) -> OnlineResetDetector:
    algorithm = ShewhartControlChart(
        learning_period_size=LEARNING_PERIOD_SIZE,
        window_size=WINDOW_SIZE,
    )
    return OnlineResetDetector(
        algorithm,
        threshold=float(threshold),
        skip_period=SKIP_PERIOD,
        max_runlength=MAX_RUNLENGTH,
        collect_states=True,
        data_transformer=feature_transformer,
    )


def make_entries() -> list[OnlineResetBenchmarkEntry]:
    return [OnlineResetBenchmarkEntry(make_detector(threshold)) for threshold in THRESHOLDS]


def detector_threshold(description) -> float:
    return float(description.parameters["threshold"])

In [4]:
dataset = preset_dataset(BENCHMARK_PRESET, n_series=N_SERIES, seed=SEED)
entries = make_entries()
benchmark = OnlineResetBenchmark(dataset=dataset, registry=BenchmarkRegistry())

print(f"Preset: {BENCHMARK_PRESET}")
print(f"Series count: {len(dataset)}")
print(f"Feature: {FEATURE}")
print(f"Thresholds: {THRESHOLDS.tolist()}")
print(f"Skip period: {SKIP_PERIOD}")

## Interactive Dataset Preview

In [6]:
series_options = [(provider.name, idx) for idx, provider in enumerate(dataset)]
series_dropdown = Dropdown(options=series_options, description="Series")


def render_series_preview(series_idx: int) -> None:
    plt.close("all")
    provider = dataset[series_idx]
    values = provider.dataset()[FEATURE].to_numpy(dtype=np.float64)

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(values, color="tab:blue", linewidth=1.0, label=FEATURE)
    for cp in provider.change_points:
        ax.axvline(cp, color="tab:red", linestyle="--", alpha=0.8)
    ax.set_title(f"{provider.name} | change_points={list(provider.change_points)}")
    ax.set_xlabel("index")
    ax.set_ylabel(FEATURE)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")
    fig.show()


preview_out = interactive_output(render_series_preview, {"series_idx": series_dropdown})
VBox([series_dropdown, preview_out])

## Benchmark Metrics

The same benchmark registry is reused across metrics. The first metric call registers detector runs; later metric calls reuse matching registry entries unless `force_recompute=True` is passed.

In [6]:
def classification_rows(metric_values: dict) -> list[dict]:
    rows = []
    for description, values in metric_values.items():
        row = {
            "threshold": detector_threshold(description),
            "skip_period": description.parameters["skip_period"],
            "max_runlength": description.parameters["max_runlength"],
        }
        row.update(values)
        rows.append(row)
    return rows


def scalar_rows(metric_values: dict, metric_name: str) -> list[dict]:
    return [
        {"threshold": detector_threshold(description), metric_name: value}
        for description, value in metric_values.items()
    ]

In [7]:
%%capture
classification_values = benchmark.get_metrics_for(entries, ClassificationReport(error_margin=ERROR_MARGIN), n_jobs=4)
arl_values = benchmark.get_metrics_for(entries, ARLMetric())
mean_delay_values = benchmark.get_metrics_for(entries, MeanDelayMetric(max_delay=MAX_DELAY))
median_delay_values = benchmark.get_metrics_for(entries, MedianDelayMetric(max_delay=MAX_DELAY))

classification_table = pd.DataFrame(classification_rows(classification_values))
arl_table = pd.DataFrame(scalar_rows(arl_values, "arl"))
mean_delay_table = pd.DataFrame(scalar_rows(mean_delay_values, "mean_delay"))
median_delay_table = pd.DataFrame(scalar_rows(median_delay_values, "median_delay"))

metrics_table = (
    classification_table.merge(arl_table, on="threshold")
    .merge(mean_delay_table, on="threshold")
    .merge(median_delay_table, on="threshold")
    .sort_values("threshold")
    .reset_index(drop=True)
)

In [8]:
metrics_table

## Metric Plots By Threshold

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

axes[0, 0].plot(metrics_table["threshold"], metrics_table["precision"], marker="o", label="precision")
axes[0, 0].plot(metrics_table["threshold"], metrics_table["recall"], marker="o", label="recall")
axes[0, 0].set_title("Precision and recall")
axes[0, 0].set_xlabel("threshold")
axes[0, 0].legend()

axes[0, 1].plot(metrics_table["threshold"], metrics_table["f1"], marker="o", color="tab:green")
axes[0, 1].set_title("F1")
axes[0, 1].set_xlabel("threshold")

axes[1, 0].plot(metrics_table["threshold"], metrics_table["arl"], marker="o", color="tab:purple")
axes[1, 0].set_title("Average run length")
axes[1, 0].set_xlabel("threshold")

axes[1, 1].plot(metrics_table["threshold"], metrics_table["mean_delay"], marker="o", label="mean delay")
axes[1, 1].plot(metrics_table["threshold"], metrics_table["median_delay"], marker="o", label="median delay")
axes[1, 1].set_title("Delay")
axes[1, 1].set_xlabel("threshold")
axes[1, 1].legend()

for ax in axes.ravel():
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
plt.close(fig)

## Single-Run Trace Inspection

The benchmark registry already contains the detector-provider runs from the metric calls above. The preview below retrieves the cached run for the selected series and threshold, then uses `OnlineCpdPlotter` to show the same trace that contributed to the benchmark table.

In [7]:
trace_series_dropdown = Dropdown(options=series_options, description="Series")
threshold_dropdown = Dropdown(
    options=[(f"{threshold:.2f}", float(threshold)) for threshold in THRESHOLDS],
    value=float(THRESHOLDS[len(THRESHOLDS) // 2]),
    description="Threshold",
)


def render_trace(series_idx: int, threshold: float) -> None:
    plt.close("all")
    provider = dataset[series_idx]
    selected_entry = next(entry for entry in entries if np.isclose(detector_threshold(entry.description), threshold))
    registry_items = list(benchmark.registry.executions_registry.items())
    matching_items = [
        (descr, single_run)
        for descr, single_run in registry_items
        if descr.detector_description == selected_entry.description
        and descr.provider_description.name == provider.annotation.name
    ]
    if not matching_items:
        raise LookupError(f"No cached run found for {provider.annotation.name} at threshold={threshold:.2f}")

    _, run = matching_items[0]
    transformed_provider = feature_transformer.transform(run.provider)
    trace = run.trace

    plotter = OnlineCpdPlotter(
        backend=DrawBackend.MATPLOTLIB,
        data_provider=transformed_provider,
        detection_trace=trace,
    )
    plotter.set_ground_truth(list(run.provider.change_points), margin=ERROR_MARGIN[1])
    plotter.set_legend_axis("detection_function")
    plotter.timeseries_visualizer.set_plot_opts(title=f"{provider.name} | threshold={threshold:.2f}", ylabel=FEATURE)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    plotter.draw(
        figure=fig,
        axes={"timeseries": axes[0], "detection_function": axes[1]},
    )
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        if labels:
            unique = dict(zip(labels, handles, strict=False))
            ax.legend(unique.values(), unique.keys(), loc="upper right")
    plt.tight_layout()
    fig.show()


trace_out = interactive_output(
    render_trace,
    {"series_idx": trace_series_dropdown, "threshold": threshold_dropdown},
)
VBox([trace_series_dropdown, threshold_dropdown, trace_out])